# 212. RAG 父子切块：怎样兼顾精确召回、上下文和引用？

> **面试问题：怎样设计 child 索引、parent hydrate、ACL/版本校验、双层引用与 child-recall/parent-sufficiency 评测？**

## 先给结论

不要只背论文名或框架名。应当说明输入/状态合同、核心算法、失败分支、独立 oracle、指标和可回滚制品。以下均使用受控小数据验证实现机制；真实生产仍需替换模型、权限、索引、安全审计与线上评测。

## 一手资料

- [RAG](https://arxiv.org/abs/2005.11401)
- [H-RAG Parent-Child Retrieval](https://arxiv.org/abs/2605.00631)
- [Adaptive Chunking](https://arxiv.org/abs/2603.25333)

In [ ]:
contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "versioned"}  # 执行本行的状态、计算或校验逻辑。
assert contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert contract["production"] == "versioned"  # 执行本行的状态、计算或校验逻辑。
assert len(contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 父文档与 child 合同

child 用于精确命中，parent 用于生成时保留完整条件。每个 child 必须有 parent id、内容范围、ACL、版本和索引版本；否则更新/删除、权限和引用无法保持一致。


In [ ]:
parents = {"p1": {"text": "已发货订单需要人工确认后退款", "acl": "support", "version": 2}, "p2": {"text": "普通订单三天送达", "acl": "support", "version": 1}}  # 执行本行的状态、计算或校验逻辑。
children = [{"id": "c1", "parent": "p1", "text": "已发货订单需要人工确认"}, {"id": "c2", "parent": "p1", "text": "确认后退款"}, {"id": "c3", "parent": "p2", "text": "普通订单三天送达"}]  # 执行本行的状态、计算或校验逻辑。
assert len(parents) == 2  # 执行本行的状态、计算或校验逻辑。
assert all(item["parent"] in parents for item in children)  # 执行本行的状态、计算或校验逻辑。
assert parents["p1"]["version"] == 2  # 执行本行的状态、计算或校验逻辑。


## 2. child 一致性

切块器必须保持 child 内容来自对应 parent，且继承相同 ACL。真实实现还要记录 token offset、标题路径、chunker/overlap 版本，避免同一文本在不同索引策略下被混用。


In [ ]:
def valid_child(child):  # 执行本行的状态、计算或校验逻辑。
    parent = parents.get(child["parent"])  # 执行本行的状态、计算或校验逻辑。
    return parent is not None and child["text"] in parent["text"] and parent["acl"] == "support"  # 执行本行的状态、计算或校验逻辑。
assert all(valid_child(item) for item in children)  # 执行本行的状态、计算或校验逻辑。
assert not valid_child({"id": "bad", "parent": "x", "text": "a"})  # 执行本行的状态、计算或校验逻辑。
assert len({item["id"] for item in children}) == 3  # 执行本行的状态、计算或校验逻辑。


## 3. 细粒度召回

教学以词重叠实现 child ranking；生产可替换为 BM25、dense、hybrid 和 rerank。候选阶段仍只能输出真实 child 和分数，不能把生成文本误当成检索证据。


In [ ]:
def score(query, text):  # 执行本行的状态、计算或校验逻辑。
    return len(set(query.replace(" ", "")) & set(text))  # 执行本行的状态、计算或校验逻辑。
def retrieve(query, top_k):  # 执行本行的状态、计算或校验逻辑。
    ranked = sorted(children, key=lambda item: score(query, item["text"]), reverse=True)  # 执行本行的状态、计算或校验逻辑。
    return [item for item in ranked if score(query, item["text"]) > 0][:top_k]  # 执行本行的状态、计算或校验逻辑。
hits = retrieve("人工确认 退款", 2)  # 执行本行的状态、计算或校验逻辑。
assert [item["id"] for item in hits] == ["c1", "c2"]  # 执行本行的状态、计算或校验逻辑。
assert all(item["parent"] == "p1" for item in hits)  # 执行本行的状态、计算或校验逻辑。
assert retrieve("不存在", 2) == []  # 执行本行的状态、计算或校验逻辑。


## 4. parent hydrate

多个 child 命中同一个 parent 时，直接拼接会重复。hydrate 应去重、执行 ACL 检查，并在父级/Token 预算内回填最少必要上下文；生产应谨慎处理不相关 sibling 的噪声。


In [ ]:
def hydrate(hits, role, budget):  # 执行本行的状态、计算或校验逻辑。
    selected = []  # 执行本行的状态、计算或校验逻辑。
    seen = set()  # 执行本行的状态、计算或校验逻辑。
    for child in hits:  # 执行本行的状态、计算或校验逻辑。
        parent = parents[child["parent"]]  # 执行本行的状态、计算或校验逻辑。
        if child["parent"] not in seen and parent["acl"] == role and len(selected) < budget:  # 执行本行的状态、计算或校验逻辑。
            selected.append({"parent": child["parent"], "text": parent["text"], "version": parent["version"]})  # 执行本行的状态、计算或校验逻辑。
            seen.add(child["parent"])  # 执行本行的状态、计算或校验逻辑。
    return selected  # 执行本行的状态、计算或校验逻辑。
context = hydrate(hits, "support", 2)  # 执行本行的状态、计算或校验逻辑。
assert len(context) == 1  # 执行本行的状态、计算或校验逻辑。
assert context[0]["parent"] == "p1"  # 执行本行的状态、计算或校验逻辑。
assert "人工确认" in context[0]["text"]  # 执行本行的状态、计算或校验逻辑。


## 5. 双层引用

答案的 evidence 应保存既命中的 child，又实际提供给模型的 parent。只给 parent 标题无法说明精确依据；只给 child 又可能遗漏生成可见的条件。两层 id 与版本使引用可复放。


In [ ]:
def citations(hits, context):  # 执行本行的状态、计算或校验逻辑。
    allowed = {item["parent"] for item in context}  # 执行本行的状态、计算或校验逻辑。
    return [{"child": item["id"], "parent": item["parent"], "version": parents[item["parent"]]["version"]} for item in hits if item["parent"] in allowed]  # 执行本行的状态、计算或校验逻辑。
evidence = citations(hits, context)  # 执行本行的状态、计算或校验逻辑。
assert [item["child"] for item in evidence] == ["c1", "c2"]  # 执行本行的状态、计算或校验逻辑。
assert all(item["parent"] == "p1" for item in evidence)  # 执行本行的状态、计算或校验逻辑。
assert all(item["version"] == 2 for item in evidence)  # 执行本行的状态、计算或校验逻辑。


## 6. 陈旧索引失败

parent 内容、权限或版本变化时，旧 child embedding 可能仍被召回。hydrate 前必须比对 parent version/tombstone；不匹配应拒绝并触发重建，而不是返回陈旧但看似相关的答案。


In [ ]:
def fresh(indexed_version, parent):  # 执行本行的状态、计算或校验逻辑。
    return indexed_version == parent["version"]  # 执行本行的状态、计算或校验逻辑。
assert fresh(2, parents["p1"])  # 执行本行的状态、计算或校验逻辑。
assert not fresh(1, parents["p1"])  # 执行本行的状态、计算或校验逻辑。
assert not fresh(2, {**parents["p1"], "version": 3})  # 执行本行的状态、计算或校验逻辑。


## 7. 分层评测

child Recall@k 与 parent context sufficiency 是不同指标。评测集应标注 gold child、gold parent、引用正确和端到端答案；只优化 child recall 可能把完整条件和 ACL 错误带进生成。


In [ ]:
def metrics(retrieved, gold_children, hydrated, gold_parent):  # 执行本行的状态、计算或校验逻辑。
    recall = len(set(retrieved) & set(gold_children)) / len(gold_children)  # 执行本行的状态、计算或校验逻辑。
    return recall, gold_parent in hydrated  # 执行本行的状态、计算或校验逻辑。
recall, parent_ok = metrics([item["child"] for item in evidence], ["c1", "c2"], [item["parent"] for item in context], "p1")  # 执行本行的状态、计算或校验逻辑。
assert recall == 1.0  # 执行本行的状态、计算或校验逻辑。
assert parent_ok is True  # 执行本行的状态、计算或校验逻辑。
assert metrics([], ["c1"], [], "p1") == (0.0, False)  # 执行本行的状态、计算或校验逻辑。


## 8. 版本制品

chunker、overlap、embedding/index、parent snapshot、ACL 和评测集必须原子发布。任一变化都会影响召回、引用、成本与回滚，因此不可只更新向量库而不更新 parent 元数据。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"splitter": "sentence-v1", "children": 3, "parents": 2, "index": "idx-v1"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["children"] == 3  # 执行本行的状态、计算或校验逻辑。
assert artifact["parents"] == 2  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整答案应先定义成功条件，再说明数据状态、主路径、失败边界和评测。受控断言只证明实现不变量，不能直接外推为真实大语料、模型语义、线上成本或安全效果。
